In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

# **Flow for this notebook**

1.   Create Simulator
2.   Bitstring Generator to determine **Alice's Message and Basis** and **Bob's Basis**.
3.   Qubit Generation (Sending)
4.   Eve Interception
4.   Bob Receive
4.   Sifting (To make the private key and bob's measured key for QBER)
5.   QBER for tamper checking
6.   Additional code to see if all interceptions will be detected
1.   Different type of attack (Entangling attack)



# **Create Simulator**

In [3]:
simulator = AerSimulator()
n_bits = 16

# **Generate Bitstrings for Alice and Bob**

In [4]:
def generate_bits_quantum(n_bits): # HELPER QUANTUM RANDOMNESS to generate bit and basis

  qc = QuantumCircuit(1, 1)
  qc.h(0)
  qc.measure(0, 0)

  result = simulator.run(qc, shots=n_bits, memory=True).result() # run n_bits times with shots
  bitstring = ''.join(result.get_memory()) # combine together

  return bitstring

# Make Alice and Bob
alice_basis=generate_bits_quantum(n_bits)
alice_state=generate_bits_quantum(n_bits)
bob_basis=generate_bits_quantum(n_bits)

print("Randomly generating Alice and Bob...\n")
print(f"alice_basis: {alice_basis}")
print(f"alice_state: {alice_state}")
print(f"bob_basis: {bob_basis}")

Randomly generating Alice and Bob...

alice_basis: 0111101001111010
alice_state: 1101100111110011
bob_basis: 0101111001011101


# **Function for Alice and Bob to Send and Receive**

In [5]:
"""
LEGEND for BASIS
0 -> standard basis
1 -> diagonal basis

"""

# For Alice (and Eve), returns the circuit (Qubit)
def sender_function(state, basis):
  bitlength = len(state)

  circuit = QuantumCircuit(bitlength)

  for i in range(bitlength):
    # qubit default is ket 0
    if state[i] == '1': # if bit 1, use pauli-x gate to change to ket 1
      circuit.x(i)

    # after that, check the basis of the bit
    if basis[i] == '1': # if bit 1, use hadamard gate to change into ket - or +
      circuit.h(i)

  return circuit

# For Bob (and Eve), returns the measured bitstring
def receiver_function(message, measurement_basis):
  bitlength = len(measurement_basis)

  for i in range(bitlength):

    if measurement_basis[i] == '1': # Based on his basis, hadamard gate if 1, else leave it as it is.
      message.h(i)

  message.measure_all()

  # Run the measured circuit
  compiled_circuit = transpile(message, simulator)

  result = simulator.run(
        compiled_circuit,
        shots=1,
        memory=True
    ).result()

  return result.get_memory()[0][::-1]

# **Function for Eve to Intercept**

In [6]:
# For Eve (Attacker)
def attacker_function(message, length):
  eve_basis = generate_bits_quantum(length) # Generate basis

  eve_result = receiver_function(message, eve_basis) # Use the receiver function to measure the circuit based on own basis
  eve_resent_circuit = sender_function(eve_result, eve_basis) # Resend the message

  return eve_resent_circuit, eve_basis, eve_result

# Create qubit circuit based on Alice bits and basis
alice_encoded_circuit = sender_function(alice_state, alice_basis)

# Attack in the middle!
eve_resent_circuit, eve_basis, eve_state = attacker_function(
    alice_encoded_circuit,
    16
)

# Bob receives the tampered circuit
bob_state = receiver_function(eve_resent_circuit, bob_basis)

# **Sifting Function** : Generates both sender and receiver key

In [7]:
# To create private key + receiver key
def sifting_function(sender_basis, sender_state, receiver_basis, receiver_result):
  sender_key = "" # Private Key
  receiver_key = "" # Subset using receiver_state (To calculate QBER)

  for i in range(len(sender_basis)):
    if sender_basis[i] == receiver_basis[i]:
      sender_key += sender_state[i]
      receiver_key += receiver_result[i]

  return sender_key, receiver_key

# Generate Keys
sifted_key, receiver_key = sifting_function(alice_basis, alice_state, bob_basis, bob_state)

# **Check Quantum Bit Error Rate**

In [8]:
# Identify mismatched bits within the matched basis to check for tampering
def calculate_qber(sender_key, receiver_key):

  if len(sender_key) == 0: # if 0
    return 0, 0

  errors = 0

  # count the number of different bits between private key and receiver state
  for i in range(len(sender_key)):
    if sender_key[i] != receiver_key[i]:
      errors += 1

  # length of key divided by number of errors
  qber = errors / len(sender_key)

  return qber, errors

# Calculate QBER
qber, errors = calculate_qber(sifted_key, receiver_key)
print(f"Quantum Bit Error Rate: {qber}")
print(f"Number of errors: {errors}")

Quantum Bit Error Rate: 0.3
Number of errors: 3


Final Results

In [9]:
# Final Conclusion Block

print("\n==============================")
print("BB84 Protocol with Attacker Summary")
print("==============================")

print("\n Generated the bits and basis for both Alice and Bob using Quantum Randomness")
print("\n Results as shown: ")
print("----------------------------------")

print("\nAlice basis: ", alice_basis)
print("Alice state: ", alice_state)
print("Bob basis:   ", bob_basis)

print("\nEve basis: ", eve_basis)
print("Eve state: ", eve_state)

print("\n Sifting to generate key")
print("----------------------------------")

print("Sifted key (From Alice's Bit): ", sifted_key)
print("Receiver key (From Bob's Measured Bit): ", receiver_key)
print("Sifted key length: ", len(sifted_key))

print("\n QBER Checking to see if message had been tampered")
print("----------------")
print("Number of errors:", errors)
print("QBER:", qber)
print("QBER percentage:", qber * 100, "%")

print("\n Attack Detection")
print("----------------------------------")

if (qber > 0.11): # Following standard of 11% threshold
  print("Attack detected!")
else:
  print("No attack detected.")



BB84 Protocol with Attacker Summary

 Generated the bits and basis for both Alice and Bob using Quantum Randomness

 Results as shown: 
----------------------------------

Alice basis:  0111101001111010
Alice state:  1101100111110011
Bob basis:    0101111001011101

Eve basis:  1111111010001110
Eve state:  1101110110100011

 Sifting to generate key
----------------------------------
Sifted key (From Alice's Bit):  1111011110
Receiver key (From Bob's Measured Bit):  1111010000
Sifted key length:  10

 QBER Checking to see if message had been tampered
----------------
Number of errors: 3
QBER: 0.3
QBER percentage: 30.0 %

 Attack Detection
----------------------------------
Attack detected!


Additional to test if security can be breached.



In [10]:
def run_bb84(n_bits):
    # Generate Alice and Bob random bits/basis
    alice_basis = generate_bits_quantum(n_bits)
    alice_state = generate_bits_quantum(n_bits)
    bob_basis = generate_bits_quantum(n_bits)

    # Alice sends, Bob receives
    alice_qubit = sender_function(alice_state, alice_basis)

    eve_resent_circuit, eve_basis, eve_state = attacker_function(
    alice_qubit,
    16
    )

    bob_state = receiver_function(eve_resent_circuit, bob_basis)

    # Sifting
    sifted_key, receiver_key = sifting_function(
        alice_basis,
        alice_state,
        bob_basis,
        bob_state
    )

    # QBER
    qber, errors = calculate_qber(sifted_key, receiver_key)

    return {
        "alice_basis": alice_basis,
        "alice_state": alice_state,
        "bob_basis": bob_basis,
        "bob_state": bob_state,
        "eve_basis": eve_basis,
        "eve_state": eve_state,
        "sifted_key": sifted_key,
        "receiver_key": receiver_key,
        "sifted_key_length": len(sifted_key),
        "qber": qber,
        "errors": errors
    }

def test_attack_detection_percentage(n_bits, trials):
    attack_detected_count = 0
    qber_values = []

    for i in range(trials):
        result = run_bb84(n_bits)

        qber = result["qber"]
        qber_values.append(qber)

        if qber > 0.11:
            attack_detected_count += 1

    attack_detection_percentage = (attack_detected_count / trials) * 100
    average_qber = sum(qber_values) / trials

    print("\n==============================")
    print("BB84 Attack Detection Test")
    print("==============================")

    print(f"Number of bits per run: {n_bits}")
    print(f"Number of trials: {trials}")
    print(f"QBER threshold: 0.11")

    print("\nResults")
    print("----------------------------------")
    print(f"Number of attacks detected: {attack_detected_count}")
    print(f"Attack detection percentage: {attack_detection_percentage:.2f}%")
    print(f"Average QBER: {average_qber:.4f}")
    print(f"Average QBER percentage: {average_qber * 100:.2f}%")

    print("\nTheory Check")
    print("----------------------------------")
    if attack_detection_percentage > 0:
        print("Attack was detected in some runs.")
    else:
        print("No attack was detected in any run.")

    return qber_values

In [11]:
test_attack_detection_percentage(16, 50)


BB84 Attack Detection Test
Number of bits per run: 16
Number of trials: 50
QBER threshold: 0.11

Results
----------------------------------
Number of attacks detected: 42
Attack detection percentage: 84.00%
Average QBER: 0.2243
Average QBER percentage: 22.43%

Theory Check
----------------------------------
Attack was detected in some runs.


[0.2,
 0.0,
 0.25,
 0.375,
 0.3,
 0.2,
 0.25,
 0.3333333333333333,
 0.2,
 0.3,
 0.1111111111111111,
 0.0,
 0.4,
 0.2857142857142857,
 0.125,
 0.09090909090909091,
 0.2857142857142857,
 0.18181818181818182,
 0.25,
 0.16666666666666666,
 0.16666666666666666,
 0.42857142857142855,
 0.2222222222222222,
 0.36363636363636365,
 0.09090909090909091,
 0.25,
 0.18181818181818182,
 0.0,
 0.14285714285714285,
 0.4444444444444444,
 0.18181818181818182,
 0.21428571428571427,
 0.36363636363636365,
 0.5,
 0.0,
 0.09090909090909091,
 0.3333333333333333,
 0.3,
 0.1111111111111111,
 0.0,
 0.2222222222222222,
 0.1111111111111111,
 0.2222222222222222,
 0.375,
 0.14285714285714285,
 0.2727272727272727,
 0.5714285714285714,
 0.25,
 0.23076923076923078,
 0.125]

# **Entangling Attack**

A different kind of attack, in which Eve tries to get more information

Simplified version using CNOT gate

In [12]:
# For Eve (Attacker) (Entangle Version)
def entangle_attacker_function(message, length):

    # Create a larger circuit:
    # First half  = Alice's qubits sent to bob
    # Second half = Eve's hidden qubits
    circuit = QuantumCircuit(length * 2, length * 2)

    # Place Alice's encoded message into the first half of the circuit
    circuit.compose(
        message.copy(),
        qubits=list(range(length)),
        inplace=True
    )

    # Eve entangles each Alice qubit with her own qubits
    for i in range(length):
        alice_qubit = i
        eve_qubit = i + length

        # CNOT links alice's qubit to eve's qubits
        # Qubits carry information about alice's qubit
        circuit.cx(alice_qubit, eve_qubit)

    # sent back to bob
    return circuit

# For Bob and Eve (Entangle Version)
def receiver_function_entangle_bob_and_eve(message, bob_basis, alice_basis):
    bitlength = len(bob_basis)

    measured_message = message.copy() # for code purpose

    # Bob measures Alice's travelling qubits only
    # These are qubits 0 to bitlength - 1
    for i in range(bitlength):

        if bob_basis[i] == '1':
            measured_message.h(i)

        measured_message.measure(i, i)

    # Eve measures her hidden qubits later
    for i in range(bitlength):
        eve_qubit = i + bitlength
        eve_classical_bit = i + bitlength

        # Eve uses Alice's basis after basis announcement
        if alice_basis[i] == '1':
            measured_message.h(eve_qubit)

        # Measure eve's hidden qubits
        measured_message.measure(eve_qubit, eve_classical_bit)

    compiled_circuit = transpile(measured_message, simulator)

    result = simulator.run(
        compiled_circuit,
        shots=1,
        memory=True
    ).result()

    measured_bits = result.get_memory()[0][::-1]

    bob_state = measured_bits[:bitlength] # bob
    eve_state = measured_bits[bitlength:] # eve

    return bob_state, eve_state

In [13]:
# Create new basis and state
alice_basis=generate_bits_quantum(14) # chose 14 to fit restraints
alice_state=generate_bits_quantum(14)
bob_basis=generate_bits_quantum(14)

# Send the message
alice_encoded_circuit = sender_function(alice_state, alice_basis)

# Eve entangles Alice's qubits with her own hidden qubits
entangled_circuit = entangle_attacker_function(
    alice_encoded_circuit,
    14 # smaller to fit restraints
)

# Bob receives it, while eve waits for it again..
bob_state, eve_state = receiver_function_entangle_bob_and_eve(
    entangled_circuit,
    bob_basis,
    alice_basis
)

# Sifting for Alice and Bob
sifted_key, receiver_key = sifting_function(
    alice_basis,
    alice_state,
    bob_basis,
    bob_state
)

# Sifting Eve's result using the same Alice/Bob matched basis positions
_, eve_key = sifting_function(
    alice_basis,
    alice_state,
    bob_basis,
    eve_state
)

# Calculate QBER
qber, errors = calculate_qber(sifted_key, receiver_key)

print("Alice basis:", alice_basis)
print("Alice state:", alice_state)
print("Bob basis:  ", bob_basis)

print("\nBob state:", bob_state)
print("Eve state:", eve_state)

print("\nSifted key: ", sifted_key)
print("Receiver key:", receiver_key)
print("Eve key:     ", eve_key)

print("\nQBER:", qber)
print("QBER percentage:", qber * 100, "%")
print("Number of errors:", errors)

if qber > 0.11:
    print("Attack detected!")
else:
    print("No attack detected.")

Alice basis: 01000110011000
Alice state: 11111000001101
Bob basis:   10000001000100

Bob state: 10111010001101
Eve state: 11111010001101

Sifted key:  111001
Receiver key: 111001
Eve key:      111001

QBER: 0.0
QBER percentage: 0.0 %
Number of errors: 0
No attack detected.


Compared to intercept-resend, the entangling attack is more subtle because Eve does not directly destroy and recreate the qubits. But because we use CNOT to entangle alice's into eve's, if the basis is diagonal, there is a chance that it can mess up the circuit. Hence, its possible that the attack will be detected